# Week 02 — BBO capstone driver

Round 2. One observation per function is not enough to fit anything useful, so this round is still exploration — but this time each function gets its own spread point rather than a slice of a shared vector.

**Design intent:** cover regions W1 did not touch, with no two coordinates deliberately correlated. The aim is a second anchor per function far enough from the first that the pair says something about scale.

In hindsight this is the most productive round of the project — it sets the best-ever value for F6, F7 and F8, the three hardest functions, and nothing in the following eleven rounds beats any of them.

In [ ]:
%matplotlib inline
import os, sys, warnings
warnings.filterwarnings("ignore")
# Walk up until bbo.py is found, so the notebook runs from anywhere in the repo.
_root = os.getcwd()
while not os.path.exists(os.path.join(_root, "bbo.py")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
os.chdir(_root); sys.path.insert(0, _root)
import numpy as np
import pandas as pd
import bbo

WEEK = 2
PRIOR = WEEK - 1          # data state this round was proposed from
SEED = 2
OUTDIR = f"outputs/week{WEEK:02d}"; os.makedirs(OUTDIR, exist_ok=True)

# What each function is getting this round, and why.
PLAN = {
    1: 'broad probe, low corner',
    2: 'broad probe, high x2',
    3: 'broad probe, extreme coords',
    4: 'broad probe, mid-cube',
    5: 'broad probe, low x1',
    6: 'broad probe, mid-cube',
    7: 'broad probe, low corner',
    8: 'broad probe, mixed',
}
pd.DataFrame([dict(func=f"F{f}", d=bbo.DIMS[f], move=PLAN[f]) for f in bbo.FUNC_IDS])


## 1. Data — the state this round was proposed from

Best point on record per function, truncated to rounds ≤ 1. Nothing below this cell may look at later rounds.


In [ ]:
# The ledger comes FIRST every round: the best point on record, not the latest one.
led = bbo.ledger(up_to=PRIOR)
led["best"] = led["best"].map(lambda v: f"{v:.6g}")
led["x"] = led["x"].map(bbo.submission)
led


## 2. Proposals — independent spread points

Each vector chosen by hand to sit well away from the W1 point in its own space.

In [ ]:
proposals = {
    1: np.array([0.018957, 0.259878]),
    2: np.array([0.098559, 0.954719]),
    3: np.array([0.159998, 0.011915, 0.958587]),
    4: np.array([0.611147, 0.607958, 0.671974, 0.601141]),
    5: np.array([0.014852, 0.297741, 0.718557, 0.219953]),
    6: np.array([0.398747, 0.385554, 0.570014, 0.711777, 0.389141]),
    7: np.array([0.151858, 0.148558, 0.071547, 0.258484, 0.285157, 0.741141]),
    8: np.array([0.159174, 0.118198, 0.137956, 0.716535, 0.781515, 0.543548, 0.279585, 0.258543]),
}

# How far is each from last round's probe? A pair too close carries no new information.
pd.DataFrame([dict(func=f"F{fid}",
                   dist_from_W1=round(float(np.linalg.norm(
                       proposals[fid] - np.array(bbo.HISTORY[1][fid][0]))), 4),
                   submission=bbo.submission(proposals[fid]))
              for fid in bbo.FUNC_IDS])


### Surrogate trust check

Run before reading any acquisition value, not after.


In [ ]:
# Is each surrogate worth listening to? LOO R2 < 0 means it is worse than
# predicting the mean, and any acquisition value built on it is arbitrary.
rows = []
for fid in bbo.FUNC_IDS:
    X, y, _ = bbo.load(fid, up_to=PRIOR)
    r2 = bbo.fit(fid, up_to=PRIOR).loo_r2() if len(y) >= 4 else float("nan")
    rows.append(dict(func=f"F{fid}", n_data=len(y), loo_r2=round(r2, 3),
                     verdict="broken" if r2 < 0 else "usable" if r2 == r2 else "too few points"))
pd.DataFrame(rows)


### Anchor audit


In [ ]:
ANCHOR = {
    1: [0.156843, 0.587493],
    2: [0.156843, 0.587493],
    3: [0.156843, 0.587493, 0.741585],
    4: [0.156843, 0.587493, 0.741585, 0.985632],
    5: [0.156843, 0.587493, 0.741585, 0.985632],
    6: [0.156843, 0.587493, 0.741585, 0.985632, 0.148524],
    7: [0.156843, 0.587493, 0.741585, 0.985632, 0.148524, 0.179638],
    8: [0.156843, 0.587493, 0.741585, 0.985632, 0.148524, 0.179638, 0.541596, 0.396585],
}
# Anchor audit: is each proposal being generated from the best point on record?
# This is the check whose absence cost the campaign most of its final score.
for fid in bbo.FUNC_IDS:
    w = bbo.anchor_check(fid, np.array(ANCHOR[fid], float), up_to=PRIOR)
    print(f"F{fid}: {w if w else 'anchored on best-known point'}")


## 3. Visualise

Best-so-far trajectory per function, truncated to the data available this round.


In [ ]:
import matplotlib
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, fid in zip(axes.ravel(), bbo.FUNC_IDS):
    try:
        _, y, rounds = bbo.load(fid, up_to=PRIOR)
    except ValueError:
        ax.set_title(f"F{fid}: no data"); continue
    ax.plot(rounds, y, "o", ms=4, alpha=.55)
    ax.plot(rounds, np.maximum.accumulate(y), "-", lw=2)
    ax.set_title(f"F{fid} (d={bbo.DIMS[fid]})", fontsize=9)
    ax.tick_params(labelsize=7); ax.set_xlabel("round", fontsize=8)
fig.suptitle(f"Best so far through round {PRIOR}", fontsize=11)
fig.tight_layout(); fig.savefig(f"{OUTDIR}/trajectories.png", dpi=140)
plt.show()


## 4. Submission strings


In [ ]:
# Portal format: six decimals, dash-separated, one line per function, no labels.
for fid in bbo.FUNC_IDS:
    print(bbo.submission(proposals[fid]))


## 5. After the portal returns each y

Returns recorded below and folded into `bbo.HISTORY` so the next round sees them.


In [ ]:
# Week 2 portal returns - already folded into bbo.HISTORY.
# returned_y = {
#     1: -1.7422969321162947e-131,
#     2: 0.0032941097425224245,
#     3: -0.323441538925546,
#     4: -11.656144229892409,
#     5: 26.446551474663757,
#     6: -0.3982565809200795,
#     7: 2.4237613938590665,
#     8: 9.6444395995596,
# }
#
# Three best-ever values set here and never beaten: F6 -0.398, F7 2.424, F8 9.644.
# F7 in particular sits at a low corner, nowhere near the domain centre - a fact
# that later rounds assumed away.
#
# for fid, y in returned_y.items():
#     bbo.append_result(fid, proposals[fid], y, rnd=WEEK)
